# LimeSurvey Analysis Notebook

This notebook loads and explores LimeSurvey exports.

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')

In [48]:
# Update filename if needed
csv_path = 'results-survey432293.csv'
df = pd.read_csv(csv_path)

print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

Rows: 11, Columns: 135


,id,submitdate,lastpage,startlanguage,seed,startdate,datestamp,Q00001[SQ001],Q00001[SQ002],Q00001[SQ003],...,groupTime5717,Q00030Time,Q00031Time,Q00032Time,Q00033Time,Q00034Time,groupTime5708,Q00035Time,Q00036Time,Q00037Time
0,2,2026-03-25 11:03:14,9,en,1851706079,2026-03-25 10:44:51,2026-03-25 11:03:14,No,Yes,No,...,20.58,NaN,NaN,NaN,NaN,NaN,5.30,NaN,NaN,NaN
1,4,2026-03-25 11:06:59,9,en,663332136,2026-03-25 10:45:13,2026-03-25 11:06:59,No,Yes,No,...,36.65,NaN,NaN,NaN,NaN,NaN,64.01,NaN,NaN,NaN
2,6,2026-03-25 11:04:24,9,en,386207750,2026-03-25 10:45:17,2026-03-25 11:04:24,No,Yes,No,...,35.19,NaN,NaN,NaN,NaN,NaN,3.16,NaN,NaN,NaN
3,7,2026-03-25 11:02:06,9,en,1234118210,2026-03-25 10:45:43,2026-03-25 11:02:06,No,No,No,...,15.19,NaN,NaN,NaN,NaN,NaN,56.62,NaN,NaN,NaN
4,8,2026-03-25 11:05:12,9,en,1444895015,2026-03-25 10:46:41,2026-03-25 11:05:12,No,Yes,No,...,26.27,NaN,NaN,NaN,NaN,NaN,72.29,NaN,NaN,NaN


# Cleanup unused columns

In [49]:
# change the index to the id column

df = df.set_index("id")

In [50]:
# drop all columns starting with Q00007..Q00011
# (intro questions + object comprehension) no need for that

prefixes = [f"Q{i:05d}" for i in range(7, 12)]
#text field
prefixes.append("Q00026")
prefixes.append("Q00014")
prefixes.append("Q00030")


cols_to_drop = [
    col for col in df.columns
    if any(col.startswith(prefix) for prefix in prefixes)
]
print(cols_to_drop)
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns")

['Q00007', 'Q00008', 'Q00009', 'Q00010', 'Q00011[SQ001]', 'Q00011[SQ002]', 'Q00011[SQ003]', 'Q00011[SQ004]', 'Q00011[SQ005]', 'Q00014', 'Q00026', 'Q00030', 'Q00007Time', 'Q00008Time', 'Q00009Time', 'Q00010Time', 'Q00011Time', 'Q00014Time', 'Q00026Time', 'Q00030Time']
Dropped 20 columns


# Creating Subsets


Expertise Level Description
Aviation Domain Expert Active or former Air Traffic Controller
Aviation Semi-Domain Expert Person working or researching on ATM in general
Aviation Layperson Any other person without sufficient knowledge about
ATM or ATC

Expertise Level Description
AI Domain Expert Studied or researches AI-relevant subjects and has
knowledge about XAI and RL
AI Semi-Domain Expert Person having an understanding of AI but not about
XAI
AI Layperson Any other person without sufficient knowledge about


In [51]:

atc_knowledge_levels = [
    "Working Knowledge: I understand ATC procedures and operational context in depth (e.g., work in ATM, studied it formally)",
    "Expert: I have direct operational experience as a controller or deep professional ATC expertise",
    "Conceptual Understanding: I understand key ATC concepts, terminology, and challenges without operational experience",
]

mask_domain = df["Q00001[SQ001]"].eq("Yes")  # active/former ATC
mask_semibase = df["Q00001[SQ002]"].eq("Yes")  # ATM-related work/research
mask_knowledge = df["Q00002"].isin(atc_knowledge_levels)

# Make groups disjoint:
mask_semi = (~mask_domain) & mask_semibase & mask_knowledge
mask_layman = ~(mask_domain | mask_semi)

AviationDomainExperts = df[mask_domain]
AviationSemiDomainExperts = df[mask_semi]
AviationLayman = df[mask_layman]

# Optional check:
print(len(AviationDomainExperts), len(AviationSemiDomainExperts), len(AviationLayman), len(df))


0 4 7 11


In [52]:


ml_expert_knowledge_levels = [
    "Technical: I understand core mechanics (e.g., gradient descent, loss functions, model architectures) and can work with them confidently",
    "Expert: I have deep theoretical and practical ML knowledge, including mathematical foundations, and can critically analyze or develop models"
]
ml_semi_knowledge_levels = [
    "Applied: I can use ML tools/libraries without fully understanding the math",
    "Technical: I understand core mechanics (e.g., gradient descent, loss functions, model architectures) and can work with them confidently",
    "Expert: I have deep theoretical and practical ML knowledge, including mathematical foundations, and can critically analyze or develop models"
]

mask_ai_domain_experts = (df["Q00001[SQ004]"].eq("Yes") | df["Q00001[SQ003]"].eq("Yes"))  # Student or aiml reaearch professional
mask_rl_xrl = (df["Q00004"].eq("Yes") & df["Q00005"].eq("Yes")) # knowledge a bout XRL RL
mask_not_xai = df["Q00005"].eq("No")

mask_expert_knowledge = df["Q00003"].isin(ml_expert_knowledge_levels)
mask_semi_knowledge = df["Q00003"].isin(ml_semi_knowledge_levels)

mask_expert = (mask_ai_domain_experts & mask_rl_xrl & mask_expert_knowledge)
mask_semi = (~mask_expert & mask_semi_knowledge)
mask_layman = ~(mask_expert | mask_semi)

AiDomainExperts = df[mask_expert]
AiSemiDomainExperts = df[mask_semi]
AiLayman= df[mask_layman]

print(len(AiDomainExperts), len(AiSemiDomainExperts), len(AiLayman), len(df))


0 7 4 11


In [57]:
groups = {
    "All Participants": df,
    "AI Domain Experts": AiDomainExperts,
    "AI Semi-Domain Experts": AiSemiDomainExperts,
    "AI Layman": AiLayman,
    "Aviation Domain Experts": AviationDomainExperts,
    "Aviation Semi-Domain Experts": AviationSemiDomainExperts,
    "Aviation Layman": AviationLayman,
}


In [58]:
# Question-code pairs for pre/post comparison

mental_model_pairs = {
    "saliency_safe_state": {"pre": "Q00012", "post": "Q00022"},
    "saliency_conflict_state": {"pre": "Q00013", "post": "Q00021"},
    "moe": {"pre": "Q00015", "post": "Q00024"},
    "action_heatmaps": {"pre": "Q00016", "post": "Q00025"},
}

explanation_trust_pairs = {
    "trust_item_1": {"pre": "Q00017", "post": "Q00031"},
    "trust_item_2": {"pre": "Q00018", "post": "Q00032"},
    "trust_item_3": {"pre": "Q00019", "post": "Q00033"},
    "trust_item_4": {"pre": "Q00020", "post": "Q00034"},
}

question_pairs = {
    "mental_models": mental_model_pairs,
    "explanation_trust": explanation_trust_pairs,
}

likert_scale = {"I agree strongly": 2,
                "I agree somewhat": 1,
                "I’m neutral about it": 0,
                "I disagree somewhat": -1,
                "I disagree strongly": -2
                
                }

question_pairs

{'mental_models': {'saliency_safe_state': {'pre': 'Q00012', 'post': 'Q00022'},
  'saliency_conflict_state': {'pre': 'Q00013', 'post': 'Q00021'},
  'moe': {'pre': 'Q00015', 'post': 'Q00024'},
  'action_heatmaps': {'pre': 'Q00016', 'post': 'Q00025'}},
 'explanation_trust': {'trust_item_1': {'pre': 'Q00017', 'post': 'Q00031'},
  'trust_item_2': {'pre': 'Q00018', 'post': 'Q00032'},
  'trust_item_3': {'pre': 'Q00019', 'post': 'Q00033'},
  'trust_item_4': {'pre': 'Q00020', 'post': 'Q00034'}}}

In [ ]:
def calculate_cronbach_alpha(df):
    """Calculates Cronbach's alpha for a dataframe of items."""
    k = df.shape[1]
    if k < 2:
        return np.nan
        
    item_variances = df.var(ddof=1)
    total_variance = df.sum(axis=1).var(ddof=1)
    
    if total_variance == 0:
        return 0.0
        
    alpha = (k / (k - 1)) * (1 - (item_variances.sum() / total_variance))
    return alpha

# Pre post trust comparison


In [73]:
# ...existing code...
def compare_pre_post(df, question_pairs, likert_scale, section="explanation_trust"):
    pairs = question_pairs[section]

    rows = []
    per_person = pd.DataFrame(index=df.index)

    for item, cols in pairs.items():
        pre = df[cols["pre"]].map(likert_scale)
        post = df[cols["post"]].map(likert_scale)
        delta = post - pre

        per_person[f"{item}_pre"] = pre
        per_person[f"{item}_post"] = post
        per_person[f"{item}_delta"] = delta

        rows.append({
            "item": item,
            "n_paired": int((pre.notna() & post.notna()).sum()),
            "pre_mean": pre.mean(),
            "post_mean": post.mean(),
            "delta_mean": delta.mean(),
        })

    item_summary = pd.DataFrame(rows).set_index("item")

    pre_cols = [c for c in per_person.columns if c.endswith("_pre")]
    post_cols = [c for c in per_person.columns if c.endswith("_post")]
    delta_cols = [c for c in per_person.columns if c.endswith("_delta") and c != "overall_delta"]

    
    per_person["overall_pre"] = per_person[pre_cols].mean(axis=1, skipna=True)
    per_person["overall_post"] = per_person[post_cols].mean(axis=1, skipna=True)
    per_person["overall_delta"] = per_person["overall_post"] - per_person["overall_pre"]

    overall_summary = pd.Series({
        "n_people_with_any_pair": int(per_person["overall_delta"].notna().sum()),
        "overall_pre_mean": per_person["overall_pre"].mean(),
        "overall_post_mean": per_person["overall_post"].mean(),
        "overall_delta_mean": per_person["overall_delta"].mean(),
        "cronbach_alpha_pre": calculate_cronbach_alpha(per_person[pre_cols]),
        "cronbach_alpha_post": calculate_cronbach_alpha(per_person[post_cols]),
        "cronbach_alpha_delta": calculate_cronbach_alpha(per_person[delta_cols]),
    })

    return item_summary, overall_summary, per_person




In [74]:
from IPython.display import display, Markdown


all_results = {}

for group_name, gdf in groups.items():
    item_summary, overall_summary, scores = compare_pre_post(
        gdf, question_pairs, likert_scale, section="explanation_trust"
    )
    all_results[group_name] = {
        "item_summary": item_summary,
        "overall_summary": overall_summary,
        "scores": scores,
    }

    display(Markdown(f"### {group_name} (n={len(gdf)})"))
    display(item_summary)
    display(overall_summary)

### All Participants (n=11)

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,11,0.272727,0.454545,0.181818
trust_item_2,11,0.454545,0.454545,0.000000
trust_item_3,11,0.181818,0.272727,0.090909
trust_item_4,11,0.272727,0.363636,0.090909


n_people_with_any_pair    11.000000
overall_pre_mean           0.295455
overall_post_mean          0.386364
overall_delta_mean         0.090909
cronbach_alpha_pre         0.413392
cronbach_alpha_post        0.650318
cronbach_alpha_delta       0.101644
dtype: float64

### AI Domain Experts (n=0)

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,0,NaN,NaN,NaN
trust_item_2,0,NaN,NaN,NaN
trust_item_3,0,NaN,NaN,NaN
trust_item_4,0,NaN,NaN,NaN


...

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,4,1.00,0.75,-0.25
trust_item_2,4,0.25,0.50,0.25
trust_item_3,4,1.00,0.50,-0.50
trust_item_4,4,0.50,0.50,0.00


n_people_with_any_pair    4.000000
overall_pre_mean          0.687500
overall_post_mean         0.562500
overall_delta_mean       -0.125000
cronbach_alpha_pre       -0.842105
cronbach_alpha_post       0.592593
cronbach_alpha_delta     -0.095238
dtype: float64

### Aviation Layman (n=7)

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,7,-0.142857,0.285714,0.428571
trust_item_2,7,0.571429,0.428571,-0.142857
trust_item_3,7,-0.285714,0.142857,0.428571
trust_item_4,7,0.142857,0.285714,0.142857


n_people_with_any_pair    7.000000
overall_pre_mean          0.071429
overall_post_mean         0.285714
overall_delta_mean        0.214286
cronbach_alpha_pre        0.533333
cronbach_alpha_post       0.666667
cronbach_alpha_delta      0.333333
dtype: float64

Explanation Satisfaction comparison

In [68]:
questions ={
    "object_based_saliency": "Q00027",
    "action_heatmap": "Q00028",
    "moe_explanation": "Q00029"   
}

subquestions = [f"SQ{i:03d}" for i in range(1, 7)] 
print(subquestions)

['SQ001', 'SQ002', 'SQ003', 'SQ004', 'SQ005', 'SQ006']


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown



def analyze_explanation_satisfaction(df, questions, subquestions, likert_scale):
    results = {}
    for q_name, q_code in questions.items():
        satisfaction_cols = [f"{q_code}[{sq}]" for sq in subquestions]
        satisfaction_data = df[satisfaction_cols].apply(lambda col: col.map(likert_scale.get))
        
        # Calculate Cronbach's alpha BEFORE adding the mean column
        alpha = calculate_cronbach_alpha(satisfaction_data[satisfaction_cols])
        
        # Calculate mean across subquestions for each respondent
        satisfaction_data[f"{q_name}_mean"] = satisfaction_data.mean(axis=1, skipna=True)
        
        # Calculate correlation between subquestions and mean
        correlations = {}
        for col in satisfaction_cols:
            corr = satisfaction_data[col].corr(satisfaction_data[f"{q_name}_mean"])
            correlations[col] = corr
        
        results[q_name] = {
            "item_data": satisfaction_data,
            "overall_mean": satisfaction_data[f"{q_name}_mean"].mean(),
            "overall_median": satisfaction_data[f"{q_name}_mean"].median(),
            "n_responses": int(satisfaction_data[f"{q_name}_mean"].notna().sum()),
            "cronbach_alpha": alpha,
            "correlations": correlations,
        }
    
    return results

# Run and display results
# (Assuming df, questions, subquestions, and likert_scale are defined above)
satisfaction_results = analyze_explanation_satisfaction(df, questions, subquestions, likert_scale)

for q_name, q_results in satisfaction_results.items():
    display(Markdown(f"### {q_name}"))
    display(q_results["item_data"])
    
    summary_metrics = {
        "overall_mean": q_results["overall_mean"],
        "overall_median": q_results["overall_median"],
        "n_responses": q_results["n_responses"],
        "cronbach_alpha": q_results["cronbach_alpha"]
    }
    display(pd.Series(summary_metrics))
    
    display(Markdown("Correlations with mean:"))
    display(pd.Series(q_results["correlations"]))

### object_based_saliency

,Q00027[SQ001],Q00027[SQ002],Q00027[SQ003],Q00027[SQ004],Q00027[SQ005],Q00027[SQ006],object_based_saliency_mean
id,,,,,,,
2,1,1,1,1,1,1,1.000000
4,0,1,-1,0,1,-1,0.000000
6,1,0,0,-1,0,1,0.166667
7,0,0,0,0,0,0,0.000000
8,1,1,2,2,-2,1,0.833333
12,2,0,-1,-2,-1,1,-0.166667
20,2,0,1,0,0,2,0.833333
21,1,-1,-1,-1,-1,-1,-0.666667
24,1,-1,-2,-2,0,-1,-0.833333


overall_mean       0.287879
overall_median     0.166667
n_responses       11.000000
cronbach_alpha     0.751282
dtype: float64

Correlations with mean:

Q00027[SQ001]    0.251617
Q00027[SQ002]    0.841507
Q00027[SQ003]    0.918635
Q00027[SQ004]    0.803614
Q00027[SQ005]    0.249815
Q00027[SQ006]    0.780858
dtype: float64

...

### moe_explanation

,Q00029[SQ001],Q00029[SQ002],Q00029[SQ003],Q00029[SQ004],Q00029[SQ005],Q00029[SQ006],moe_explanation_mean
id,,,,,,,
2,1,1,1,1,1,1,1.000000
4,0,-1,-1,-1,0,0,-0.500000
6,-2,-2,-2,-2,-2,-2,-2.000000
7,0,0,0,0,0,0,0.000000
8,1,1,1,1,1,-1,0.666667
12,-1,-2,1,1,-1,0,-0.333333
20,2,2,1,2,1,1,1.500000
21,2,1,0,1,1,0,0.833333
24,1,0,-2,0,-1,-1,-0.500000


overall_mean       0.212121
overall_median     0.000000
n_responses       11.000000
cronbach_alpha     0.944436
dtype: float64

Correlations with mean:

Q00029[SQ001]    0.854694
Q00029[SQ002]    0.904877
Q00029[SQ003]    0.840548
Q00029[SQ004]    0.924814
Q00029[SQ005]    0.944567
Q00029[SQ006]    0.852450
dtype: float64

Write that the alpha is generall high enough to justify the mean as a measure central tendency. Also interestingly for some explanations the alpha is higher.